# Train Franok on Colab
Select a GPU runtime. This notebook fits the bilingual tokenizer, initializes every parameter from scratch, trains the model and exports browser weights. The reference run used an NVIDIA L4. Save the output directory before disconnecting.

In [ ]:
%cd /content
!git clone https://github.com/yunusemrejr/Franok.git
%cd /content/Franok
%pip install -q -r requirements.txt

In [ ]:
%env HF_HUB_DISABLE_XET=1
%env RAYON_NUM_THREADS=4
!python -m training.data

In [ ]:
!python -m training.train --steps 3200 --batch 24 --length 256 --minutes 65

In [ ]:
!python -m training.refine --pretrain 4800 --sft 1200 --batch 24 --minutes 35

In [ ]:
!python -m training.export
!ls -lh work/run/model.safetensors work/export/model.q4

The first release is in work/run. The following cell continues this same local scratch lineage through the extended release. It prepares additional datasets, increases the context to 1,024, performs supervised and low-rank tuning, samples replies for reinforcement learning, and exports Q4 and Q8. The default extended training stages reserve about 108 minutes in addition to data preparation and file transfer. Copy work/extended before disconnecting.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "training.extended.pipeline", "--scratch-run", "work/run", "--tokenizer", "work/data/tokenizer.json"], check=True)

The optional third run continues those same scratch-trained weights on English and Turkish web text and stories, followed by isolated supervised conversations, two LoRAs and on-policy reinforcement learning. Select a fresh four-hour L4 allocation before enabling this cell. The pretraining samples are drawn from both languages with equal probability by token position. All stage receipts and checkpoints are written to `work/third/run`.

In [ ]:
run_third_stage = False  # Set True to allocate up to four additional hours.
if run_third_stage:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "training.third.pipeline", "--minutes", "240", "--parent", "work/extended/final/model.safetensors", "--tokenizer", "work/data/tokenizer.json"], check=True)

### Export the third run on CPU

Keep the trained run, prepared dialogue directories and the original tokenizer. After the GPU training stages finish, these commands reproduce the Q4 and Q8 browser files with a CPU runtime. The release uses the unrounded reinforcement merge. Calibration uses training examples. Evaluation uses separate held-out examples.

```bash
python -m training.calibration --run work/third/run/fp32 --out work/third/calibration.npz --threads 3 --paired-groups
python -m training.export --run work/third/run/fp32 --out work/third/export --tokenizer work/data/tokenizer.json --bits 4 --calibration work/third/calibration.npz --full-range --covariance --paired-groups
python -m training.export --run work/third/run/fp32 --out work/third/export --tokenizer work/data/tokenizer.json --bits 8
```
